# Conversational Agentic System

This is the complete Studio system without a UI. A deterministic Python Agent bounds conversation context; a reasoning Agent uses the provider and framework selected by `.env`. The same builder powers the Streamlit app.


## 1. Configuration contract

In ADA, copy the bundle-root `.env.example` to the bundle-root `.env`; that is the only runtime configuration file. No credential is entered in the notebook and no provider fallback is enabled. Set `RUN_STUDIO_LIVE=1` when the selected provider is ready.\n

In [ ]:
import os

import agentic_systems as toolkit
from agentic_systems_studio import (
    ConversationConfig,
    build_conversational_system,
    load_studio_environment,
    safe_calculate,
)

environment_path = load_studio_environment()
config = ConversationConfig.from_environment()
RUN_STUDIO_LIVE = (
    os.getenv('RUN_STUDIO_LIVE', '0').lower() in {'1', 'true', 'yes'}
    and os.getenv('AGENTIC_SYSTEMS_NOTEBOOK_TEST') != '1'
)
toolkit.show_json({
    'environment_path': str(environment_path),
    'provider': config.provider,
    'framework': config.framework,
    'model': config.model,
    'run_live': RUN_STUDIO_LIVE,
}, title='Studio runtime contract')


## 2. Deterministic evidence

The Tool runs without an LM. This verifies the deterministic boundary before we compile the reasoning runtime.


In [ ]:
calculation = safe_calculate.run({'expression': '17 * 19'})
assert calculation.ok and calculation.data['result'] == 323
toolkit.show_json(calculation.data, title='Deterministic Tool evidence')


## 3. Build and converse

`build_conversational_system` is provider/framework-agnostic. The result must retain the selected runtime identity and pass common invariants.


In [ ]:
if RUN_STUDIO_LIVE:
    studio = build_conversational_system(config)
    result = studio.run('Explain why 17 * 19 is 323 and use the calculator.')
    assert result.ok, result.errors
    result.check_invariants()
    toolkit.human_result(result, title='Conversational Studio RunResult')
else:
    studio = None
    result = None
    toolkit.show_json({'status': 'not-run', 'reason': 'RUN_STUDIO_LIVE=0'})


## Acceptance

The run is accepted when the declared provider/framework executes without fallback, deterministic tool evidence is observable and the final `RunResult` passes invariants.
